# Train / Test 


The split is chronological: the test set is the last 20% of the rows and we don't touch it until the end. The test set starts on 2025-05-16 (144 rows, train has 578).

The target doesn't look ahead anymore (it ends at the close of day `t`), so we don't need to remove rows of train. We keep `gap = 1` in `TimeSeriesSplit`.

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
data = pd.read_feather("../data/final_data_ny.feather")

In [ ]:
n_test = int(len(data) * 0.2)
TEST_START = data.index[-n_test]
train = data.loc[data.index < TEST_START]
test = data.loc[TEST_START:]

In [ ]:
gap = 1

In [ ]:
print("train:", train.index.min(), "->", train.index.max(), len(train))
print("test: ", test.index.min(), "->", test.index.max(), len(test))
print("TEST_START:", TEST_START.date())

In [ ]:
test.to_feather("../data/test_holdout_ny.feather")

Class balance: in train the NY session goes up 55.5% of the days (56.9% in test). So a model that always says "up" already gets that accuracy, that's the baseline to beat.

In [ ]:
target_cols = [("ret_ny", ""), ("y_ny", "")]
feature_cols = [c for c in data.columns if c not in target_cols]

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, gap=gap)
X_train = train.drop(columns=target_cols)
y_train = train[[("y_ny", "")]]

## Checks

In [ ]:
print("final shape:", data.shape)
print("train:", train.shape, train.index.min().date(), "->", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "->", test.index.max().date())

In [ ]:
print([c for c in feature_cols if c[0] != "stock_ret"])
print((data.columns.get_level_values(0) == "stock_ret").sum(), "stock_ret columns")

In [ ]:
print(train[("y_ny", "")].value_counts(normalize=True))
print(test[("y_ny", "")].value_counts(normalize=True))

Leakage sanity check: correlation of each feature with the target return on train (NQ features first, then the daily features). A value above 0.3 (in absolute value) would be suspicious.

In [ ]:
# the NQ overnight features (built in 1_Project_Features)
nq_names = ["asia_ret", "europe_ret", "ovn_ret", "asia_high_dist", "asia_low_dist",
            "europe_high_dist", "europe_low_dist", "ovn_high_dist", "ovn_low_dist", "ovn_range_pos",
            "prev_ny_high_dist", "prev_ny_low_dist", "ovn_volume_rel", "asia_volume_share",
            "europe_volume_share", "last_hour_volume_rel", "ovn_vol", "ovn_range"]
nq_cols = [(n, "") for n in nq_names]
assert all(c in data.columns for c in nq_cols)

In [ ]:
corr = train[nq_cols].corrwith(train[("ret_ny", "")])
corr.index = [c[0] for c in corr.index]
print(corr.sort_values())
print("max |corr|:", corr.abs().max())

In [ ]:
daily_cols = [c for c in feature_cols if c[0] != "stock_ret" and c not in nq_cols]
corr_daily = train[daily_cols].corrwith(train[("ret_ny", "")])
corr_daily.index = [c[0] for c in corr_daily.index]
print(corr_daily.sort_values())
print("max |corr|:", corr_daily.abs().max())

**Known limitation:** the table has one row per day and the NQ file only covers Dec 2022 - Dec 2025, so we only have 722 rows (578 for train and 144 for test). It is small. Possible options for later: a longer NQ source (since 2008, not on Kaggle), or a panel with one row per stock per day.